In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_producaoSMDcomapontamento AS (

--CTE COM OS DADOS DE PRODUÇÂO
WITH BASE_PRODUCAO_SLA_2026 AS (
    SELECT   
        fca.OrdemProducao,
        fap.CodProdutoAcabado AS CodProd,
        fca.SerieProdutoAcabado AS Serie,
        CAST(fa.DataHoraApontamento AS DATE) AS DataProducao,
        fa.DataHoraApontamento,
        SUM(CAST(fap.QuantidadeApontada AS INT)) AS Qtd_Produzida
    FROM gold.sankhya.fato_apontamento_produto fap
    INNER JOIN gold.sankhya.fato_apontamento fa ON fap.CodApontamentoUnico = fa.CodApontamentoUnico
    INNER JOIN gold.sankhya.fato_controle_apontamento fca ON fa.CodApontamentoUnico = fca.CodApontamentoUnico
    WHERE fa.DataHoraApontamento IS NOT NULL
      AND try_cast(fa.DataHoraApontamento AS DATE) BETWEEN '2025-01-01' AND CURRENT_DATE()
    GROUP BY fca.OrdemProducao, fap.CodProdutoAcabado, fca.SerieProdutoAcabado, fa.DataHoraApontamento
),
-- CTE COM OS DADOS DE REPARO
REPAROS_CONSOLIDADO AS (
    SELECT 
        cab.OrdemProducao, 
        cab.SerieProdutoAcabado AS Serie,
        COUNT(DISTINCT cab.CodLancamentoReparo) AS Qtd_Reparos,
        SUM(CAST(item.QuantidadeNegociada AS INT)) AS Pecas_Substituidas,
        COALESCE(par.NomeParceiro, 'Não Informado') AS Tecnico,
        CONCAT_WS(', ', COLLECT_SET(sint.DescricaoSintoma)) AS Sintomas,
        CONCAT_WS(', ', COLLECT_SET(def.DescricaoDefeito)) AS Defeitos,
        CONCAT_WS(', ', COLLECT_SET(acao.DescricaoAcaoCorretiva)) AS Acoes
    FROM gold.sankhya.fato_producao_reparo_cabecalho cab
    INNER JOIN gold.sankhya.fato_producao_reparo_item item ON cab.CodLancamentoReparo = item.CodLancamentoReparo
    LEFT JOIN gold.sankhya.dim_producao_sintoma_reparo sint ON cab.CodSintoma = sint.CodSintoma
    LEFT JOIN gold.sankhya.dim_producao_defeito_reparo def ON item.CodDefeito = def.CodDefeito
    LEFT JOIN gold.sankhya.dim_producao_acao_reparo acao ON item.CodAcaoCorretiva = acao.CodAcaoCorretiva
    LEFT JOIN gold.sankhya.dim_parceiros par
    ON try_cast(cab.CodUsuarioOperador AS INT) = par.CodParceiro
    WHERE CAST(cab.DataHoraInicioReparo AS DATE) BETWEEN '2025-01-01' AND CURRENT_DATE()
    GROUP BY cab.OrdemProducao, cab.SerieProdutoAcabado, par.NomeParceiro
),
-- CTE COM OS DADOS DE EXPEDIÇÃO
DADOS_EXPEDICAO AS (
    SELECT
        CodProduto,
        SUM(QtdNegociada) AS QtdExpedida
    FROM gold.sankhya.fato_itens
    WHERE DataFaturamento >= '2026-01-01'
    GROUP BY CodProduto
)
--SUBQUERY FINAL
SELECT
    'CONTAGEM/MG' AS Planta,
    fca_pct.DescricaoPosto AS CentroTrabalho,
    PROD.OrdemProducao AS OP,
    PROD.Serie,
    PROD.DataProducao, 
    PROD.DataHoraApontamento,
    MONTH(PROD.DataHoraApontamento) AS Mes,
    YEAR(PROD.DataHoraApontamento) AS Ano,
    PROD.CodProd, 
    SKU.DescricaoProduto,
    CONCAT(CAST(PROD.CodProd AS STRING), ' - ', COALESCE(SKU.DescricaoProduto, '')) AS Descricao_SKU,
    SKU.Marca AS Fornecedor,
    SKU.ModeloMkt AS Modelo,
    GRP.NomeGrupoPai AS Familia,
    GRP.LinhaDeNegocio,
    GRP.NomeGrupoFamilia,
    SKU.UsadoComo,
    PROD.Qtd_Produzida,
    ROUND(PROD.Qtd_Produzida * COALESCE(fopi.QuantidadeAProduzir, 0)
        / NULLIF(SUM(PROD.Qtd_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd), 0)
    ) AS Qtd_Planejada,
    FLOOR(
        SUM(PROD.Qtd_Produzida) OVER (PARTITION BY PROD.CodProd ORDER BY PROD.DataHoraApontamento DESC ROWS UNBOUNDED PRECEDING)
        * COALESCE(EXP.QtdExpedida, 0)
        / NULLIF(SUM(PROD.Qtd_Produzida) OVER (PARTITION BY PROD.CodProd), 0)
    )
    - FLOOR(
        (SUM(PROD.Qtd_Produzida) OVER (PARTITION BY PROD.CodProd ORDER BY PROD.DataHoraApontamento DESC ROWS UNBOUNDED PRECEDING) - PROD.Qtd_Produzida)
        * COALESCE(EXP.QtdExpedida, 0)
        / NULLIF(SUM(PROD.Qtd_Produzida) OVER (PARTITION BY PROD.CodProd), 0)
    ) AS Qtd_Expedida,
    COALESCE(REP.Qtd_Reparos, 0) AS Qtd_Reparos,
    COALESCE(REP.Pecas_Substituidas, 0) AS Pecas_Substituidas,
    REP.Tecnico,
    COALESCE(NULLIF(REP.Sintomas, ''), 'Sem Registro') AS Sintomas,
    COALESCE(NULLIF(REP.Defeitos, ''), 'Sem Registro') AS Defeitos,
    COALESCE(NULLIF(REP.Acoes, ''), 'Sem Registro') AS Acoes
FROM BASE_PRODUCAO_SLA_2026 PROD
INNER JOIN gold.sankhya.dim_produtos SKU ON PROD.CodProd = SKU.CodProduto
LEFT JOIN gold.sankhya.dim_grupo_produtos GRP ON SKU.CodGrupoProduto = GRP.CodGrupoProduto
INNER JOIN (
    SELECT DISTINCT fca.OrdemProducao, fca.SerieProdutoAcabado, pct.DescricaoPosto
    FROM gold.sankhya.fato_controle_apontamento fca
    INNER JOIN gold.sankhya.dim_posto_trabalho pct ON fca.CodCentroTrabalho = pct.CodCentroTrabalho
    WHERE pct.DescricaoPosto IN (
        '1- QUALIDADE SMD MONTAGEM MAESTRO PLUS', 
        '3- QUALIDADE PTH MONTAGEM DE COMPONENTES', 
        '5- EMBALAGEM SMT MAESTRO PLUS'
    )
) fca_pct ON PROD.OrdemProducao = fca_pct.OrdemProducao
    AND PROD.Serie = fca_pct.SerieProdutoAcabado
LEFT JOIN gold.sankhya.fato_ordem_producao_item fopi ON PROD.OrdemProducao = fopi.OrdemProducao 
    AND PROD.CodProd = fopi.CodProdutoAcabado
LEFT JOIN REPAROS_CONSOLIDADO REP 
    ON PROD.OrdemProducao = REP.OrdemProducao 
    AND PROD.Serie = REP.Serie
LEFT JOIN DADOS_EXPEDICAO EXP
ON PROD.CodProd = EXP.CodProduto
ORDER BY PROD.DataHoraApontamento DESC
);